# LoRA SFT v7 — MoE Tying CORRECTED (per-expert separate modules) + Verification

**Why v7 over v6:** v6 (LB 0.58, flat vs v3) had a silent bug. Cell 2.5's diagnostic showed PEFT created per-expert SEPARATE LoRA modules (e.g. `experts.0.up_proj.lora_A`, `experts.1.up_proj.lora_A`, ...) with shape `(rank, dim)` — NOT a single batched `(num_experts, rank, dim)` tensor like Tong's Unsloth setup. My v6 tying logic mistakenly took `mean(dim=0)` over the rank dim, effectively downgrading every MoE expert LoRA to rank-1. The other arch changes (q/k/v/o + lm_head + alpha=r + dropout=0 + betas) compensated, netting flat at 0.58. We never actually tested MoE tying.

**v7 fixes the tying logic:**
- Group params by `(layer, projection, side)` across all expert modules — e.g. all `layers.1.mixer.experts.{0..127}.up_proj.lora_A` form one group
- Init: set every member of each group to the cross-member mean (so all start tied)
- Grad step: SUM gradients across the group, copy the sum to every member's grad

**NEW: built-in verification BEFORE training (cheap insurance against another silent bug)**
1. Post-init: assert every param within each tied group has identical values (1 sec)
2. Forward+backward on one example, then assert every param's `.grad` is identical within its group (~10 sec)
3. If either check fails → `RuntimeError` → kernel aborts after ~8 min of model load, not ~5 hr of pointless training

**All other v6 changes retained (v3 → v7):**

| Param | v3 | v7 |
|---|---|---|
| `lora_alpha` | 16 | 32 |
| `lora_dropout` | 0.05 | 0.0 |
| `target_modules` | in/out/up/down_proj (4) | q/k/v/o + in/out/up/down + lm_head (9) |
| AdamW betas | (0.9, 0.999) | (0.9, 0.95) |
| **MoE expert weight tying** | ❌ | ✅ CORRECTLY (per-expert-module grouped tying) |

**Data:** byte-identical to v3 (`training-data.jsonl`, 4681 records).

**Reference points:**
- v3 (baseline, no arch changes): **0.580**
- v6 (broken tying + other arch changes): 0.580 — the other changes offset broken-tying penalty
- Tong's progress prize: 0.850

**Expected outcomes:**
- v7 ≥ 0.62: correct MoE tying was indeed the architectural piece we were missing
- v7 ≈ 0.59-0.61: tying helps modestly; further gap requires Tong's expensive stack
- v7 ≈ 0.58: tying alone doesn't move the needle for SFT-on-our-data; ceiling effect
- v7 < 0.58: aggressive tying changes loss landscape badly — abort SFT branch

Estimated walltime: ~6 hr.

In [ ]:
# Cell 1: Setup — sitedir discovery + ptxas-blackwell workaround + seeds
import os, random, shutil, site, stat, sys, time
import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Find python_packages from attached utility scripts
for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

# ptxas-blackwell exec workaround (Triton needs an executable copy)
PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break

step(f"Setup done; seeds set to {SEED}")

In [ ]:
# Cell 2: Load model + tokenizer + apply LoRA (v6 config)
import kagglehub
import mamba_ssm  # noqa: F401
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
step("Tokenizer loaded")

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
step(f"IM_END_ID={IM_END_ID}; eos_token_id={tokenizer.eos_token_id}")

t = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
step(f"Model loaded in {time.perf_counter() - t:.0f}s, VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB")

# v6 LoRA config: alpha=r (vs v3's r/2), dropout=0 (vs v3's 0.05),
# expanded target_modules including attention proj + lm_head.
lora_cfg = LoraConfig(
    r=32,
    lora_alpha=32,                  # v3 used 16
    lora_dropout=0.0,               # v3 used 0.05
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # NEW: attention
        "in_proj", "out_proj",                    # Mamba projections (had these)
        "up_proj", "down_proj",                   # FFN/MoE projections (had these)
        "lm_head",                                # NEW: output projection
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
trainable, total = sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())
step(f"LoRA wrapped: {trainable:,} trainable / {total:,} total ({100*trainable/total:.2f}%)")

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
step("Gradient checkpointing on")

In [ ]:
# Cell 2.5: MoE expert weight tying — CORRECTED for per-expert separate modules
#
# v6 lesson: PEFT created separate LoRA modules per expert
# (experts.0.up_proj, experts.1.up_proj, ..., experts.127.up_proj) with
# shape (rank, dim). v6's tying logic mistakenly averaged across the rank
# dim, downgrading every MoE expert to rank-1.
#
# v7 fixes this by GROUPING params across experts within the same
# (layer, projection, side), then tying each group:
#   - All `layers.L.mixer.experts.{0..127}.up_proj.lora_A` form ONE group
#   - Init: copy the cross-member mean to every group member
#   - Grad step: sum gradients across the group, copy sum to each member
#
# Convention (Tinker-style):
#   - For w1-side projections (up_proj/gate_proj): tie the A side
#   - For w2-side projections (down_proj):         tie the B side

import re

# Pattern extracts (everything-up-to-experts, expert_id, everything-after).
# Example match for 'base_model.model.backbone.layers.1.mixer.experts.0.up_proj.lora_A.default.weight':
#   prefix='base_model.model.backbone.layers.1.mixer.experts'
#   expert_id='0'
#   rest='up_proj.lora_A.default.weight'
EXPERT_RE = re.compile(
    r"^(?P<prefix>.+\.experts)\.(?P<expert>\d+)\.(?P<rest>.+)$"
)

W1_NAMES = ("up_proj", "gate_up_proj", "gate_proj")
W2_NAMES = ("down_proj",)

def find_moe_tied_groups(model):
    """Group per-expert LoRA params that should be tied together.

    Returns: dict {(prefix, rest): [(name, param), ...]} where each value is
    a list of per-expert copies that should all have identical values.
    """
    groups: dict = {}
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if ".lora_" not in name: continue
        m = EXPERT_RE.match(name)
        if not m: continue
        rest = m.group("rest")
        is_w1 = any(p in rest for p in W1_NAMES)
        is_w2 = any(p in rest for p in W2_NAMES)
        is_A = ".lora_A." in rest
        is_B = ".lora_B." in rest
        if not ((is_w1 and is_A) or (is_w2 and is_B)):
            continue
        key = (m.group("prefix"), rest)
        groups.setdefault(key, []).append((name, param))
    return groups

def init_moe_groups_tied(groups):
    """Initialize each group's params to the cross-member mean."""
    with torch.no_grad():
        for key, members in groups.items():
            if len(members) <= 1: continue
            stacked = torch.stack([p.data for _name, p in members], dim=0)
            mean = stacked.mean(dim=0)
            for _name, p in members:
                p.data.copy_(mean)

def verify_groups_tied(groups, *, what="params") -> list:
    """Return list of failure messages (empty if all groups are correctly tied).

    `what` should be 'params' (checks .data) or 'grads' (checks .grad)."""
    failures = []
    for key, members in groups.items():
        if len(members) <= 1: continue
        if what == "params":
            ref = members[0][1].data
            for name, p in members[1:]:
                if not torch.equal(p.data, ref):
                    failures.append(f"params not tied — group {key[1]!r}: {name} differs from {members[0][0]}")
                    break  # one fail per group is enough
        elif what == "grads":
            ref_member = next((p for _n, p in members if p.grad is not None), None)
            if ref_member is None: continue
            ref = ref_member.grad
            for name, p in members:
                if p.grad is None: continue
                if not torch.equal(p.grad, ref):
                    failures.append(f"grads not tied — group {key[1]!r}: {name}'s grad differs")
                    break
    return failures

# Find groups
groups = find_moe_tied_groups(model)
n_groups = len(groups)
n_members = sum(len(m) for m in groups.values())
n_multi = sum(1 for m in groups.values() if len(m) > 1)
print(f"MoE tied groups: {n_groups} total, {n_multi} with >1 member, {n_members} total params across all groups")

if n_multi == 0:
    print("⚠️ No multi-member tied groups found. PEFT model layout doesn't match")
    print("   our assumptions. Proceeding WITHOUT MoE tying — other v7 changes still apply.")
    USING_MOE_TIE = False
else:
    sample_keys = list(groups.keys())[:3]
    for key in sample_keys:
        members = groups[key]
        print(f"  group {key[1]!r} (prefix={key[0][:60]}...): {len(members)} members")
    if n_groups > 3:
        print(f"  ... and {n_groups - 3} more groups")
    init_moe_groups_tied(groups)
    # Verify init
    init_failures = verify_groups_tied(groups, what="params")
    if init_failures:
        print(f"\n❌ Post-init verification FAILED ({len(init_failures)} failures):")
        for msg in init_failures[:5]:
            print(f"   {msg}")
        raise RuntimeError(f"MoE tying init failed — see {len(init_failures)} group(s) above")
    print(f"\n✓ Post-init verification PASSED: all {n_multi} groups correctly tied")
    USING_MOE_TIE = True
    step(f"MoE expert LoRAs initialized in tied state ({n_multi} groups, {n_members} params)")

# Define the gradient-tying callback (used only if USING_MOE_TIE)
from transformers import TrainerCallback

class MoEGroupTieGradsCallback(TrainerCallback):
    """For each tied group, sum gradients across members and copy back to each."""
    def __init__(self, groups):
        # Cache just the params (not names) for hot loop
        self.groups = [
            [p for _name, p in members]
            for key, members in groups.items()
            if len(members) > 1
        ]

    def on_pre_optimizer_step(self, args, state, control, **kwargs):
        with torch.no_grad():
            for members in self.groups:
                grads = [p.grad for p in members if p.grad is not None]
                if not grads: continue
                grad_sum = torch.stack(grads, dim=0).sum(dim=0)
                for p in members:
                    if p.grad is not None:
                        p.grad.copy_(grad_sum)

trainer_callbacks = []
if USING_MOE_TIE:
    trainer_callbacks.append(MoEGroupTieGradsCallback(groups))
    step("MoEGroupTieGradsCallback registered")
else:
    step("No MoE callback (tying inactive)")

In [ ]:
# Cell 2.6: Grad-tying smoke test BEFORE training (~10 sec, aborts if broken)
#
# Run one forward+backward on a synthetic 1-example batch and confirm that
# AFTER the callback's on_pre_optimizer_step runs, all params within each
# tied group have IDENTICAL gradients. If they don't, the callback is broken
# (or the param grouping is wrong) — abort here instead of wasting 5 hours.
#
# We use a minimal synthetic input (32 random tokens, simple label mask) to
# trigger backprop through enough of the model. We're not testing training
# quality — just that the tying mechanism preserves the invariant.

if USING_MOE_TIE:
    step("Running grad-tying smoke test (1 forward+backward on dummy input)...")
    model.train()

    # Tiny dummy input
    dummy_len = 32
    dummy_input_ids = torch.randint(
        0, tokenizer.vocab_size, (1, dummy_len), dtype=torch.long, device=model.device
    )
    dummy_attn = torch.ones_like(dummy_input_ids)
    dummy_labels = dummy_input_ids.clone()  # next-token-predict on the same sequence

    # Forward + backward — this populates .grad on every trainable param
    model.zero_grad(set_to_none=True)
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model(input_ids=dummy_input_ids, attention_mask=dummy_attn, labels=dummy_labels)
        loss = out.loss
    loss.backward()
    step(f"Dummy backward done, loss={loss.item():.4f}")

    # BEFORE tying — grads in each group should differ (different experts saw
    # different routing, so different signals). If they don't differ, that's
    # actually unusual (would mean PEFT shares grads internally — unlikely).
    pre_tie_failures = verify_groups_tied(groups, what="grads")
    n_groups_with_grads = sum(
        1 for _key, members in groups.items()
        if len(members) > 1 and any(p.grad is not None for _n, p in members)
    )
    print(f"Pre-tie: {len(pre_tie_failures)} of {n_groups_with_grads} groups have differing grads (expected: most/all)")

    # Manually invoke the callback's tying logic — simulates what it does
    # before each optimizer step. The Trainer hasn't been built yet, so we
    # call the underlying function directly.
    cb = trainer_callbacks[0]
    with torch.no_grad():
        for members in cb.groups:
            grads = [p.grad for p in members if p.grad is not None]
            if not grads: continue
            grad_sum = torch.stack(grads, dim=0).sum(dim=0)
            for p in members:
                if p.grad is not None:
                    p.grad.copy_(grad_sum)

    # AFTER tying — grads in each group MUST be identical
    post_tie_failures = verify_groups_tied(groups, what="grads")
    if post_tie_failures:
        print(f"\n❌ Post-tying grad verification FAILED ({len(post_tie_failures)} failures):")
        for msg in post_tie_failures[:5]:
            print(f"   {msg}")
        raise RuntimeError("Grad tying broken — aborting to save 5 hours of training")
    print(f"✓ Post-tying grad verification PASSED: all {n_groups_with_grads} non-empty groups identical")

    # Clean up dummy grads
    model.zero_grad(set_to_none=True)
    step("Grad-tying smoke test PASSED — proceeding with training")
else:
    step("Skipping grad-tying smoke test (no MoE tying active)")

In [ ]:
# Cell 3: Load training data + tokenize (v3 data — UNCHANGED for clean variable test)
import json
import glob

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

# v6 deliberately uses v3's training-data.jsonl (4681 records) so the only
# variable changed from v3 is the LoRA/training architecture. NOT v5's
# training-data-v5.jsonl (which adds synthetic transformation_rules).
CANDIDATE_PATHS = [
    "/kaggle/input/datasets/gideonnazarian/nemotron-solver-training-data/training-data.jsonl",
    "/kaggle/input/nemotron-solver-training-data/training-data.jsonl",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    hits = glob.glob("/kaggle/input/**/training-data.jsonl", recursive=True)
    if hits:
        DATA_PATH = hits[0]
if DATA_PATH is None:
    raise FileNotFoundError(
        f"training-data.jsonl not found. /kaggle/input/ contains: {os.listdir('/kaggle/input')}"
    )
step(f"Using DATA_PATH={DATA_PATH}")

records = []
with open(DATA_PATH) as f:
    for line in f:
        records.append(json.loads(line))
step(f"Loaded {len(records)} training records")

from collections import Counter
cats = Counter(r["category"] for r in records)
print("Records by category:")
for c, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {c:25s}: {n}")

MAX_SEQ_LEN = 1024

def tokenize_example(record):
    user_text = record["prompt"] + PROMPT_SUFFIX
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False, add_generation_prompt=True,
    )
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    response_text = (
        record["reasoning"] + "\n</think>\n\\boxed{" + record["answer"] + "}<|im_end|>"
    )
    response_ids = tokenizer.encode(response_text, add_special_tokens=False)
    full_ids = prompt_ids + response_ids
    if len(full_ids) > MAX_SEQ_LEN:
        return None
    labels = [-100] * len(prompt_ids) + response_ids
    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids),
    }

examples = []
skipped = 0
for rec in records:
    ex = tokenize_example(rec)
    if ex is None:
        skipped += 1
    else:
        examples.append(ex)
step(f"Tokenized {len(examples)} examples ({skipped} skipped > {MAX_SEQ_LEN} tokens)")

lens = [len(ex["input_ids"]) for ex in examples]
step(f"Token-length stats: min={min(lens)}, median={sorted(lens)[len(lens)//2]}, max={max(lens)}")

ex0 = examples[0]
print("\n=== Example 0 sanity check ===")
prompt_part = tokenizer.decode([t for t, l in zip(ex0["input_ids"], ex0["labels"]) if l == -100])
response_part = tokenizer.decode([l for l in ex0["labels"] if l != -100])
print(f"PROMPT (loss masked, last 200 chars): ...{prompt_part[-200:]!r}")
print(f"RESPONSE (training target, first 300 chars): {response_part[:300]!r}")

In [ ]:
# Cell 4: Train with HF Trainer + (optional) MoE tying callback
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

class SFTDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

def collate(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    input_ids, labels, attn = [], [], []
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    for ex in batch:
        n_pad = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [pad_id] * n_pad)
        labels.append(ex["labels"] + [-100] * n_pad)
        attn.append(ex["attention_mask"] + [0] * n_pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
    }

train_ds = SFTDataset(examples)

args = TrainingArguments(
    output_dir="/kaggle/working/lora_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="no",
    bf16=True,
    optim="adamw_torch",
    adam_beta1=0.9,
    adam_beta2=0.95,               # v3 used default 0.999; Tong uses 0.95
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    gradient_checkpointing=False,  # already enabled on model
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate,
    callbacks=trainer_callbacks,   # MoEGroupTieGradsCallback (if MoE tying active)
)

step(f"Starting training (v7: alpha=32, dropout=0, 9 target modules, MoE tied={USING_MOE_TIE} via per-expert-module grouped tying)")
trainer.train()
step("Training done")

In [ ]:
# Cell 5: Save adapter to /kaggle/working and zip submission
OUTPUT_DIR = "/kaggle/working"

# Clean up any stale adapter files from prior runs
for f in os.listdir(OUTPUT_DIR):
    if f.startswith("adapter") or f.endswith(".safetensors") or f.endswith(".bin"):
        try:
            os.remove(os.path.join(OUTPUT_DIR, f))
        except (IsADirectoryError, PermissionError):
            pass

# Save trained adapter
model.save_pretrained(OUTPUT_DIR)
step("Adapter saved")

# Move any nested files up to /kaggle/working level (Trainer may have nested under lora_output/)
import subprocess
subprocess.run("find /kaggle/working -name 'adapter*' -exec mv {} /kaggle/working/ \\;", shell=True, check=False)
subprocess.run("ls -la /kaggle/working/", shell=True, check=False)

# Zip everything for submission
subprocess.run("cd /kaggle/working && rm -rf lora_output && zip -m submission.zip adapter*", shell=True, check=True)
step("submission.zip created")

In [ ]:
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")
print("Done — ready for Submit to Competition.")